# AGENTIC AI
## SESSION 3

Assignment questions and answers


## Question 1

Mini Project: Agentic Food Delivery Assistant

Build a console-based multi-tool AI agent for a food-delivery platform
integrating technique selection, tool calling, per-session memory, and
ReAct-style reasoning.

Requirements: - Provide menu options: (1) Place Order, (2) Track Order,
(3) File Complaint, (4) Get Personalised Recommendations, and (5) Exit;
exit prints a session summary. - Each of the first four options invokes
place_order(), track_order(), file_complaint(), or
get_recommendations(). Log each tool call, arguments, and result in
session_log. Define and print an MCP-style schema (name, description,
parameters) for every tool before the menu loop. - Maintain
session_memory with orders_placed, complaints_filed, preferred_cuisine,
and interaction_count, updating it after every menu action. - Call
decide_response_strategy(action_type, memory) for every action. It
returns prompt, rag, or fine_tuned based on action type and memory;
print the strategy with the tool result. - On exit, print actions in
order, the tool and strategy used for each, and final session_memory.

**Answer:**

```python
import json

# Session Memory and Log
session_memory = {
    "orders_placed": [],
    "complaints_filed": [],
    "preferred_cuisine": None,
    "interaction_count": 0,
}

session_log = []

# MCP Tool Schemas
tools = [
    {
        "name": "place_order",
        "description": "Places a new food order.",
        "parameters": {
            "type": "object",
            "properties": {
                "item": {"type": "string"},
                "cuisine": {"type": "string"},
            },
            "required": ["item", "cuisine"],
        },
    },
    {
        "name": "track_order",
        "description": "Checks delivery status of an order.",
        "parameters": {
            "type": "object",
            "properties": {"order_id": {"type": "integer"}},
            "required": ["order_id"],
        },
    },
    {
        "name": "file_complaint",
        "description": "Logs a customer complaint.",
        "parameters": {
            "type": "object",
            "properties": {"issue": {"type": "string"}},
            "required": ["issue"],
        },
    },
    {
        "name": "get_recommendations",
        "description": "Fetches food recommendations based on cuisine preference.",
        "parameters": {
            "type": "object",
            "properties": {"cuisine": {"type": "string"}},
            "required": ["cuisine"],
        },
    },
]

# Tool Definitions
def place_order(item, cuisine):
    order_id = len(session_memory["orders_placed"]) + 1
    session_memory["orders_placed"].append({"id": order_id, "item": item, "cuisine": cuisine})
    session_memory["preferred_cuisine"] = cuisine
    return f"Order #{order_id} for '{item}' placed successfully."

def track_order(order_id):
    exists = any(o["id"] == order_id for o in session_memory["orders_placed"])
    return f"Order #{order_id} is Out for Delivery." if exists else f"Order #{order_id} not found."

def file_complaint(issue):
    complaint_id = len(session_memory["complaints_filed"]) + 1
    session_memory["complaints_filed"].append({"id": complaint_id, "issue": issue})
    return f"Complaint #{complaint_id} recorded: '{issue}'."

def get_recommendations(cuisine):
    menu = {
        "Italian": ["Margherita Pizza", "Pasta Carbonara"],
        "Indian": ["Paneer Butter Masala", "Biryani"],
        "Default": ["Burger", "Fries"]
    }
    items = menu.get(cuisine, menu["Default"])
    return f"Recommended for {cuisine}: {', '.join(items)}."

# Strategy Decision Logic
def decide_response_strategy(action_type, memory):
    if action_type == "recommendation":
        return "rag"
    elif action_type == "complaint" and len(memory["complaints_filed"]) > 0:
        return "fine_tuned"
    return "prompt"

# Print MCP Schemas
print("--- MCP Tool Schemas ---")
print(json.dumps(tools, indent=2))
print("-------------------------\n")

# Main Menu Loop
while True:
    print("\n1. Place Order\n2. Track Order\n3. File Complaint\n4. Get Personalised Recommendations\n5. Exit")
    choice = input("Select an option (1-5): ").strip()
    session_memory["interaction_count"] += 1

    if choice == "1":
        action_type = "order"
        item = input("Enter food item: ")
        cuisine = input("Enter cuisine: ")
        strategy = decide_response_strategy(action_type, session_memory)
        
        # ReAct Trace: Thought -> Action -> Observation
        print(f"[Thought] User wants to order food. Strategy chosen: {strategy}.")
        print(f"[Action] Calling place_order(item='{item}', cuisine='{cuisine}')")
        result = place_order(item, cuisine)
        print(f"[Observation] {result}")
        print(f"Strategy Used: {strategy} | Result: {result}")
        
        session_log.append({"action": "Place Order", "tool": "place_order", "args": {"item": item, "cuisine": cuisine}, "strategy": strategy, "result": result})

    elif choice == "2":
        action_type = "track"
        oid = int(input("Enter Order ID: "))
        strategy = decide_response_strategy(action_type, session_memory)
        
        print(f"[Thought] User wants order status. Strategy chosen: {strategy}.")
        print(f"[Action] Calling track_order(order_id={oid})")
        result = track_order(oid)
        print(f"[Observation] {result}")
        print(f"Strategy Used: {strategy} | Result: {result}")
        
        session_log.append({"action": "Track Order", "tool": "track_order", "args": {"order_id": oid}, "strategy": strategy, "result": result})

    elif choice == "3":
        action_type = "complaint"
        issue = input("Describe the issue: ")
        strategy = decide_response_strategy(action_type, session_memory)
        
        print(f"[Thought] User filing complaint. Strategy chosen: {strategy}.")
        print(f"[Action] Calling file_complaint(issue='{issue}')")
        result = file_complaint(issue)
        print(f"[Observation] {result}")
        print(f"Strategy Used: {strategy} | Result: {result}")
        
        session_log.append({"action": "File Complaint", "tool": "file_complaint", "args": {"issue": issue}, "strategy": strategy, "result": result})

    elif choice == "4":
        action_type = "recommendation"
        cuisine = session_memory["preferred_cuisine"] or input("Enter preferred cuisine: ")
        strategy = decide_response_strategy(action_type, session_memory)
        
        print(f"[Thought] User needs recommendations. Strategy chosen: {strategy}.")
        print(f"[Action] Calling get_recommendations(cuisine='{cuisine}')")
        result = get_recommendations(cuisine)
        print(f"[Observation] {result}")
        print(f"Strategy Used: {strategy} | Result: {result}")
        
        session_log.append({"action": "Get Recommendations", "tool": "get_recommendations", "args": {"cuisine": cuisine}, "strategy": strategy, "result": result})

    elif choice == "5":
        print("\n=== SESSION SUMMARY ===")
        print("\nAction History:")
        for idx, entry in enumerate(session_log, 1):
            print(f"{idx}. Action: {entry['action']} | Tool: {entry['tool']} | Strategy: {entry['strategy']}")
        
        print("\nFinal Session Memory:")
        print(json.dumps(session_memory, indent=2))
        break
    else:
        print("Invalid choice. Try again.")
```